In [16]:
import pandas as pd 
import numpy as np

In [36]:
import xgboost as xgb
from sklearn.feature_selection import RFE
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier
from feature_engineering import feature_encoder,feature_select_classification
from sklearn.pipeline import Pipeline
import pickle

In [17]:
df=pd.read_csv('emi_cleaned.csv')

In [18]:
df.head()

,age,gender,marital_status,education,monthly_salary,employment_type,years_of_employment,company_type,house_type,monthly_rent,...,existing_loans,current_emi_amount,credit_score,bank_balance,emergency_fund,emi_scenario,requested_amount,requested_tenure,emi_eligibility,max_monthly_emi
0,38,F,Married,Professional,82600,Private,0.9,Mid-size,Rented,20000.0,...,Yes,23700.0,660.0,303200.0,70200.0,Personal Loan EMI,850000.0,15,Not_Eligible,500.0
1,38,F,Married,Graduate,21500,Private,7.0,MNC,Family,0.0,...,Yes,4100.0,714.0,92500.0,26900.0,E-commerce Shopping EMI,128000.0,19,Not_Eligible,700.0
2,38,M,Married,Professional,86100,Private,5.8,Startup,Own,0.0,...,No,0.0,650.0,672100.0,324200.0,Education EMI,306000.0,16,Eligible,27775.0
3,58,F,Married,High School,66800,Private,2.2,Mid-size,Own,0.0,...,No,0.0,685.0,440900.0,178100.0,Vehicle EMI,304000.0,83,Eligible,16170.0
4,48,F,Married,Professional,57300,Private,3.4,Mid-size,Family,0.0,...,No,0.0,770.0,97300.0,28200.0,Home Appliances EMI,252000.0,7,Not_Eligible,500.0


In [28]:
def feature_encoder(df):
    # gender_column ()
    gender_map={
        'M':'0',
        'F':'1',
    }
    df['gender']=df['gender'].replace(gender_map)
    df['gender']=df['gender'].astype(int)


    #marital_status column
    marital_status_map={
        'Single':'0',
        'Married':'1'
    }
    df['marital_status']=df['marital_status'].replace(marital_status_map)
    df['marital_status']=df['marital_status'].astype(int)

    #education column
    education_map={
        'Unknown':'0',
        'High School':'1',
        'Post Graduate':'3',
        'Professional':'4',
        'Graduate':'2',
    }
    df['education']=df['education'].replace(education_map)
    df['education']=df['education'].astype(int)

    # company_column
    company_map={
        'Small':'0',
        'Startup':'1',
        'Mid-size':'2',
        'Large Indian':'3',
        'MNC':'4'
    }
    df['company_type']=df['company_type'].replace(company_map)
    df['company_type']=df['company_type'].astype(int)

    #existing_loan column
    existing_loan_map={
        'No':'0',
        'Yes':'1'
    }
    df['existing_loans']=df['existing_loans'].replace(existing_loan_map)
    df['existing_loans']=df['existing_loans'].astype(int)

    # one_hot_encoding for ['house_type','employment_type','emi_scenario']
    cols_to_encode=['house_type','employment_type','emi_scenario']
    df=pd.get_dummies(df,columns=cols_to_encode,prefix=cols_to_encode,dtype=int)

    #emi eligibility column
    target_map={
        'Not_Eligible':'0',
        'High_Risk':'1',
        'Eligible':'2'
    }
    df['emi_eligibility']=df['emi_eligibility'].replace(target_map)
    df['emi_eligibility']=df['emi_eligibility'].astype(int)

    return df

In [32]:
def feature_select_classification(df):
    df['expense_salary_ratio']=(df['groceries_utilities']+df['college_fees']+
                                df['current_emi_amount']+df['school_fees']+
                                df['monthly_rent']+df['other_monthly_expenses']+
                                df['travel_expenses'])/df['monthly_salary']
    
    class_df=df[['education','credit_score','monthly_salary','bank_balance',
          'emergency_fund','requested_amount','requested_tenure',
          'employment_type_Government','employment_type_Private',
          'employment_type_Self-employed','emi_scenario_E-commerce Shopping EMI',
          'emi_scenario_Education EMI','emi_scenario_Home Appliances EMI',
          'emi_scenario_Personal Loan EMI','emi_scenario_Vehicle EMI',
          'expense_salary_ratio','emi_eligibility']]
    return class_df

In [33]:
df = feature_encoder(df)
df=  feature_select_classification(df)

In [35]:
df

,education,credit_score,monthly_salary,bank_balance,emergency_fund,requested_amount,requested_tenure,employment_type_Government,employment_type_Private,employment_type_Self-employed,emi_scenario_E-commerce Shopping EMI,emi_scenario_Education EMI,emi_scenario_Home Appliances EMI,emi_scenario_Personal Loan EMI,emi_scenario_Vehicle EMI,expense_salary_ratio,emi_eligibility
0,4,660.0,82600,303200.0,70200.0,850000.0,15,0,1,0,0,0,0,1,0,1.012107,0
1,2,714.0,21500,92500.0,26900.0,128000.0,19,0,1,0,1,0,0,0,0,0.906977,0
2,4,650.0,86100,672100.0,324200.0,306000.0,16,0,1,0,0,1,0,0,0,0.413473,2
3,1,685.0,66800,440900.0,178100.0,304000.0,83,0,1,0,0,0,0,0,1,0.559880,2
4,4,770.0,57300,97300.0,28200.0,252000.0,7,0,1,0,0,0,1,0,0,1.022688,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
392899,2,649.0,32400,62000.0,32600.0,506000.0,47,0,1,0,0,0,0,1,0,1.030864,0
392900,3,712.0,49200,142200.0,38100.0,708000.0,33,0,1,0,0,0,0,1,0,0.788618,0
392901,2,676.0,25700,191600.0,39700.0,93000.0,21,0,1,0,0,0,1,0,0,0.599222,1
392902,2,784.0,47200,170400.0,45600.0,144000.0,36,0,1,0,0,0,1,0,0,0.489407,2


In [37]:
X=df.drop(columns=['emi_eligibility'])
y=df['emi_eligibility']

In [38]:
st = StandardScaler()
X_scaled=st.fit_transform(X)

X_train,X_test,y_train,y_test = train_test_split(X_scaled, y, test_size=0.2,
                                                     random_state=42)

In [39]:
def train_xgboost_c(X_train,X_test,y_train,y_test):
    xgb = XGBClassifier(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=9,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    eval_metric='logloss'
    )
    xgb.fit(X_train,y_train)
    y_pred=xgb.predict(X_test)
    accuracy=accuracy_score(y_test,y_pred)
    return xgb,accuracy

In [40]:
xgb_model,xgb_accuracy=train_xgboost_c(X_train,X_test,y_train,y_test)

In [41]:
xgb_accuracy

0.9414998536541912

In [42]:
pipline=Pipeline([
        ('class_scaler',StandardScaler()),
        ('class_model',xgb_model)
    ])
pipline.fit(X,y)
pickle.dump(pipline,open('class_model.pkl','wb'))

In [1]:
import pandas as pd
import numpy as np 
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score ,mean_absolute_error
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
import xgboost as xgb
import pickle
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

In [2]:
def feature_select_regression(df):
    df['expense_salary_ratio']=(df['groceries_utilities']+df['college_fees']+
                                df['current_emi_amount']+df['school_fees']+
                                df['monthly_rent']+df['other_monthly_expenses']+
                                df['travel_expenses'])/df['monthly_salary']
      
    reg_df=df[['education','credit_score','monthly_salary',
                 'bank_balance','emergency_fund','existing_loans',
                 'expense_salary_ratio','max_monthly_emi']]
    
    return reg_df

In [3]:
def feature_encoder(df):
    # gender_column ()
    gender_map={
        'M':'0',
        'F':'1',
    }
    df['gender']=df['gender'].replace(gender_map)
    df['gender']=df['gender'].astype(int)


    #marital_status column
    marital_status_map={
        'Single':'0',
        'Married':'1'
    }
    df['marital_status']=df['marital_status'].replace(marital_status_map)
    df['marital_status']=df['marital_status'].astype(int)

    #education column
    education_map={
        'Unknown':'0',
        'High School':'1',
        'Post Graduate':'3',
        'Professional':'4',
        'Graduate':'2',
    }
    df['education']=df['education'].replace(education_map)
    df['education']=df['education'].astype(int)

    # company_column
    company_map={
        'Small':'0',
        'Startup':'1',
        'Mid-size':'2',
        'Large Indian':'3',
        'MNC':'4'
    }
    df['company_type']=df['company_type'].replace(company_map)
    df['company_type']=df['company_type'].astype(int)

    #existing_loan column
    existing_loan_map={
        'No':'0',
        'Yes':'1'
    }
    df['existing_loans']=df['existing_loans'].replace(existing_loan_map)
    df['existing_loans']=df['existing_loans'].astype(int)

    # one_hot_encoding for ['house_type','employment_type','emi_scenario']
    cols_to_encode=['house_type','employment_type','emi_scenario']
    df=pd.get_dummies(df,columns=cols_to_encode,prefix=cols_to_encode,dtype=int)

    #emi eligibility column
    target_map={
        'Not_Eligible':'0',
        'High_Risk':'1',
        'Eligible':'2'
    }
    df['emi_eligibility']=df['emi_eligibility'].replace(target_map)
    df['emi_eligibility']=df['emi_eligibility'].astype(int)

    return df

In [6]:
data=pd.read_csv('./data/emi_cleaned.csv')
encoded_data=feature_encoder(data)
selected_features=feature_select_regression(encoded_data)
selected_features.head()

,education,credit_score,monthly_salary,bank_balance,emergency_fund,existing_loans,expense_salary_ratio,max_monthly_emi
0,4,660.0,82600,303200.0,70200.0,1,1.012107,500.0
1,2,714.0,21500,92500.0,26900.0,1,0.906977,700.0
2,4,650.0,86100,672100.0,324200.0,0,0.413473,27775.0
3,1,685.0,66800,440900.0,178100.0,0,0.559880,16170.0
4,4,770.0,57300,97300.0,28200.0,0,1.022688,500.0


In [47]:
data=pd.read_csv('emi_cleaned.csv')
encoded_data=feature_encoder(data)
selected_features=feature_select_regression(encoded_data)

X=selected_features.drop(columns=['max_monthly_emi'])
y=selected_features['max_monthly_emi']

st = StandardScaler()
X_scaled=st.fit_transform(X)

In [48]:
X_train,X_test,y_train,y_test = train_test_split(X_scaled, y, test_size=0.2,
                                                     random_state=42)

In [49]:
def train_xgboost_r(X_train,X_test,y_train,y_test):
    xgb_regressor = xgb.XGBRegressor(tree_method='hist',
                                  device='cpu',
                                #   reg_lamda=0.1,
                                 reg_alpha=0.15,
                                 n_estimators=150,
                                 min_child_weight=11,
                                 max_depth=9,
                                 learning_rate=0.1)
    xgb_regressor.fit(X_train,y_train)
    y_pred=xgb_regressor.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    return xgb_regressor,rmse 

In [50]:
xgb_model,xgb_rmse=train_xgboost_r(X_train,X_test,y_train,y_test)

In [51]:
xgb_rmse

np.float64(1417.57585136568)

In [52]:
pipline=Pipeline([
        ('reg_scaler',StandardScaler()),
        ('reg_model',xgb_model)
])
pipline.fit(X,y)
pickle.dump(pipline,open('reg_model.pkl','wb'))